In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.silver.calendar
COMMENT 'Standard calendar dimension, one row per day from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS calendar_date
)
SELECT
  CAST(date_format(calendar_date, 'yyyyMMdd') AS INT)        AS date_key,
  calendar_date,
  -- Year
  year(calendar_date)                                         AS year,
  -- Quarter
  quarter(calendar_date)                                      AS quarter,
  concat('Q', quarter(calendar_date))                         AS quarter_name,
  concat(year(calendar_date), '-Q', quarter(calendar_date))   AS year_quarter,
  -- Month
  month(calendar_date)                                        AS month,
  date_format(calendar_date, 'MMMM')                          AS month_name,
  date_format(calendar_date, 'MMM')                           AS month_short_name,
  date_format(calendar_date, 'yyyy-MM')                       AS year_month,
  CAST(date_format(calendar_date, 'yyyyMM') AS INT)           AS year_month_key,
  -- Week (ISO 8601)
  weekofyear(calendar_date)                                   AS iso_week_of_year,
  extract(YEAROFWEEK FROM calendar_date)                      AS iso_year,
  -- Day
  day(calendar_date)                                          AS day_of_month,
  dayofyear(calendar_date)                                    AS day_of_year,
  extract(DAYOFWEEK_ISO FROM calendar_date)                   AS day_of_week,        -- 1 = Monday ... 7 = Sunday
  date_format(calendar_date, 'EEEE')                          AS day_name,
  date_format(calendar_date, 'EEE')                           AS day_short_name,
  -- Flags
  extract(DAYOFWEEK_ISO FROM calendar_date) IN (6, 7)         AS is_weekend,
  extract(DAYOFWEEK_ISO FROM calendar_date) NOT IN (6, 7)     AS is_weekday,
  calendar_date = last_day(calendar_date)                     AS is_month_end,
  (year(calendar_date) % 4 = 0 AND year(calendar_date) % 100 <> 0)
    OR year(calendar_date) % 400 = 0                          AS is_leap_year,
  -- Period boundaries
  date_trunc('WEEK', calendar_date)::DATE                     AS week_start_date,
  date_add(date_trunc('WEEK', calendar_date)::DATE, 6)        AS week_end_date,
  date_trunc('MONTH', calendar_date)::DATE                    AS month_start_date,
  last_day(calendar_date)                                     AS month_end_date,
  date_trunc('QUARTER', calendar_date)::DATE                  AS quarter_start_date,
  date_sub(add_months(date_trunc('QUARTER', calendar_date)::DATE, 3), 1) AS quarter_end_date,
  date_trunc('YEAR', calendar_date)::DATE                     AS year_start_date,
  make_date(year(calendar_date), 12, 31)                      AS year_end_date,
  day(last_day(calendar_date))                                AS days_in_month
FROM dates
ORDER BY calendar_date

In [0]:
%sql
SELECT
  count(*)                         AS row_count,
  count(DISTINCT calendar_date)    AS distinct_dates,
  datediff(DATE'2030-12-31', DATE'1940-01-01') + 1 AS expected_rows,
  min(calendar_date)               AS min_date,
  max(calendar_date)               AS max_date,
  sum(CAST(is_weekend AS INT))     AS weekend_days,
  count_if(month = 2 AND day_of_month = 29) AS leap_days
FROM adb_caresync.silver.calendar

In [0]:
%sql
select * from adb_caresync.silver.calendar limit 10